# Data Pre-processing — Missing Value Analysis, Imputation & Normalisation
Covers: temporal split → missing value inspection → column dropping → leakage-free sector-median imputation → leakage-free winsorisation → leakage-free z-score standardisation.

## 1. Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import json
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('simfin_dataset.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:10]} ...")
df.head(3)


## 2. Temporal Train / Validation / Test Split (Revision 3)

> **Revision 3 — Preventing Data Leakage:**  
> The original pipeline computed imputation statistics (sector medians), winsorisation bounds, and z-score parameters on the **full dataset** before splitting. This constitutes *look-ahead bias*: statistics derived from FY2023–2024 observations leak information about future distributions into the training features.
>
> The corrected pipeline defines the temporal split **before any fitting**, computes all statistics **on the training partition only**, serialises the frozen fit objects, and applies them identically to validation and test sets.

**Temporal split:**

| Partition | Fiscal Years | Purpose |
|-----------|-------------|--------|
| Training | FY 2020–2022 | Model fitting + all preprocessing statistics |
| Validation | FY 2023 | Hyperparameter tuning |
| Test (holdout) | FY 2024 | Final evaluation only |

In [ ]:
# ── REVISION 3 — Temporal Split (must happen BEFORE any imputation or scaling) ─
# All imputation statistics, winsorisation bounds, and z-score parameters must
# be computed on the TRAINING partition only, then frozen and applied to val/test.
# Computing on the full panel before the split constitutes look-ahead bias.

TRAIN_YEARS = [2020, 2021, 2022]
VAL_YEARS   = [2023]
TEST_YEARS  = [2024]

# Ensure FiscalYear column is present (created in data_retrieval)
if 'FiscalYear' not in df.columns:
    df['Report Date'] = pd.to_datetime(df['Report Date'])
    df['FiscalYear'] = df['Report Date'].dt.year

mask_train = df['FiscalYear'].isin(TRAIN_YEARS)
mask_val   = df['FiscalYear'].isin(VAL_YEARS)
mask_test  = df['FiscalYear'].isin(TEST_YEARS)

print("Temporal split (Revision 3):")
print(f"  Train (FY {TRAIN_YEARS}): {mask_train.sum():,} rows")
print(f"  Val   (FY {VAL_YEARS}):   {mask_val.sum():,} rows")
print(f"  Test  (FY {TEST_YEARS}):  {mask_test.sum():,} rows")
print(f"  Unassigned (FY outside 2020-2024): {(~mask_train & ~mask_val & ~mask_test).sum():,} rows")


## 3. Missing Values Overview

In [ ]:
# Exclude non-feature columns from analysis
META = {'Ticker', 'Report Date', 'Sector', 'Industry', 'Perform', 'Class',
        'FiscalYear', 'fin_type', 'Company Name', 'IndustryId',
        'Market', 'Main Currency', 'Fiscal Year',
        'NBER_Recession', 'CalendarQuarter', 'PublicDate', 'MarketCap'}
feature_cols = [c for c in df.columns
                if c not in META and pd.api.types.is_numeric_dtype(df[c])]

miss_pct  = df[feature_cols].isnull().mean().sort_values(ascending=False)
miss_count = df[feature_cols].isnull().sum().sort_values(ascending=False)

print(f"Total feature columns:     {len(feature_cols)}")
print(f"Columns fully complete:    {(miss_pct == 0).sum()}")
print(f"Columns with any missing:  {(miss_pct > 0).sum()}")
print(f"Columns > 70% missing:     {(miss_pct > 0.70).sum()}")
print(f"Columns > 50% missing:     {(miss_pct > 0.50).sum()}")
print()
print(df[feature_cols].isnull().sum().sum(), "total missing cells out of",
      df[feature_cols].size, f"({df[feature_cols].isnull().mean().mean()*100:.1f}% overall)")


In [ ]:
# Full table sorted by missing rate
miss_df = pd.DataFrame({
    'Missing Count': miss_count,
    'Missing %':     (miss_pct * 100).round(1)
})
with pd.option_context('display.max_rows', 100):
    print(miss_df[miss_df['Missing Count'] > 0].to_string())


In [ ]:
# Bar chart — all columns with any missing values
to_plot = miss_pct[miss_pct > 0].sort_values()
fig, ax = plt.subplots(figsize=(12, max(4, len(to_plot) * 0.25)))
colors = ['#d62728' if v > 0.70 else '#ff7f0e' if v > 0.50 else 'steelblue'
          for v in to_plot]
to_plot.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0.70, color='red',    linestyle='--', linewidth=1.5, label='70% threshold (drop)')
ax.axvline(0.50, color='orange', linestyle='--', linewidth=1.2, label='50% threshold')
ax.set_xlabel('Missing rate')
ax.set_title('Missing rate per feature column')
ax.legend()
plt.tight_layout()
plt.show()
print("Red bars = will be dropped (>70% missing)")


In [ ]:
# Heatmap — missing pattern on a 300-row sample
sample_cols = miss_pct[miss_pct > 0].index.tolist()
if sample_cols:
    sample = df[sample_cols].sample(min(300, len(df)), random_state=42)
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(sample.isnull(), cbar=False, cmap='viridis',
                yticklabels=False, ax=ax)
    ax.set_title('Missing value pattern — 300-row sample  (yellow = missing)')
    plt.tight_layout()
    plt.show()


## 4. Drop Features with > 50% Missing Values

In [ ]:
THRESHOLD = 0.50
cols_to_drop = miss_pct[miss_pct > THRESHOLD].index.tolist()
cols_to_keep = miss_pct[miss_pct <= THRESHOLD].index.tolist()

print(f"Dropping {len(cols_to_drop)} columns with > {THRESHOLD*100:.0f}% missing:")
for c in cols_to_drop:
    print(f"  {c:<60}  {miss_pct[c]*100:.1f}%")

df_clean = df.drop(columns=cols_to_drop)
print(f"\nShape before: {df.shape}")
print(f"Shape after:  {df_clean.shape}  ({df.shape[1] - df_clean.shape[1]} columns removed)")


## 5. Impute Remaining Missing Values

In [ ]:
remaining_features = [c for c in cols_to_keep if c in df_clean.columns]

# Check how many values still need imputation
still_missing = df_clean[remaining_features].isnull().sum()
still_missing = still_missing[still_missing > 0].sort_values(ascending=False)
print(f"Columns still needing imputation: {len(still_missing)} / {len(remaining_features)}")
print(f"Total cells to impute:            {still_missing.sum():,}")
print()
print("Missing count per column (before imputation):")
print(still_missing.head(20).to_string())


In [ ]:

df_clean_train = df_clean[mask_train]   # read-only reference for fitting

imputation_fit = {}
for col in still_missing.index:
    sector_medians = {}
    if 'Sector' in df_clean_train.columns:
        sector_medians = (
            df_clean_train.groupby('Sector')[col]
            .median()
            .dropna()
            .to_dict()
        )
    global_median = float(df_clean_train[col].median())
    imputation_fit[col] = {'sector': sector_medians, 'global': global_median}

print(f"Imputation fit computed on {mask_train.sum():,} training rows.")

# Persist fit object so it can be reused / audited
with open('imputation_fit.json', 'w') as f:
    json.dump(imputation_fit, f, indent=2)
print("Imputation fit saved to imputation_fit.json")

def apply_imputation(df_in, fit, cols):
    """Apply pre-fitted imputation to a dataframe without refitting."""
    df_out = df_in.copy()
    for col in cols:
        if col not in df_out.columns:
            continue
        if fit[col]['sector'] and 'Sector' in df_out.columns:
            df_out[col] = df_out[col].fillna(
                df_out['Sector'].map(fit[col]['sector'])
            )
        df_out[col] = df_out[col].fillna(fit[col]['global'])
    return df_out

# Apply to all partitions using the SAME frozen fit
df_imputed = apply_imputation(df_clean, imputation_fit, still_missing.index)

still_after = df_imputed[remaining_features].isnull().sum().sum()
print(f"\nMissing cells before imputation: {still_missing.sum():,}")
print(f"Missing cells after imputation:  {still_after:,}")
print("All missing values imputed with no look-ahead bias." if still_after == 0
      else f"WARNING: {still_after} cells still missing.")


## 6. Final Dataset Overview

In [ ]:
print("Shape after column drop + imputation:", df_imputed.shape)
print(f"Unique companies: {df_imputed['Ticker'].nunique():,}")
print()
print("Class distribution:")
print(df_imputed['Class'].value_counts().sort_index().to_string())
print()
print("Sector breakdown:")
print(df_imputed['Sector'].value_counts().to_string())
print()
print("Sample rows:")
df_imputed[remaining_features[:6] + ['Sector','Perform','Class']].head(5)


## 7. Save Cleaned & Imputed Dataset

In [ ]:
df_imputed.to_csv('simfin_cleaned_imputed.csv', index=False)
print(f"Saved simfin_cleaned_imputed.csv")
print(f"Shape: {df_imputed.shape}")
print(f"Missing values remaining: {df_imputed[remaining_features].isnull().sum().sum()}")


## 8. Winsorisation & Z-score Standardisation

Two normalisation steps applied to the derived financial ratios before modelling:

1. **Winsorisation (1st–99th percentile)** — Clips extreme values caused by near-zero denominators (e.g., equity ≈ 0 producing arbitrarily large ROE or Leverage). Applied to all derived ratio columns.
2. **Z-score standardisation** — Centres each ratio at zero mean and scales to unit variance. Required before PCA (Section 4c) and k-means clustering.


In [ ]:
# ── Derived ratio columns ────────────────────────────────────────────────────
def safe_div(a, b):
    return a / b.replace(0, float('nan'))

df_norm = df_imputed.copy()

df_norm['ROE']             = safe_div(df_norm['Net Income (Common)'],              df_norm['Total Equity'])
df_norm['ROA']             = safe_div(df_norm['Net Income'],                       df_norm['Total Assets'])
df_norm['Leverage']        = safe_div(df_norm['Total Liabilities'],                df_norm['Total Equity'])
df_norm['GrossMargin']     = safe_div(df_norm['Gross Profit'],                     df_norm['Revenue'])
df_norm['OperatingMargin'] = safe_div(df_norm['Operating Income (Loss)'],          df_norm['Revenue'])
df_norm['CurrentRatio']    = safe_div(df_norm['Total Current Assets'],             df_norm['Total Current Liabilities'])
df_norm['DebtRatio']       = safe_div(df_norm['Total Liabilities'],                df_norm['Total Assets'])
df_norm['CFO_Margin']      = safe_div(df_norm['Net Cash from Operating Activities'], df_norm['Revenue'])


NEW_CONFOUNDER_COLS = [c for c in ['Size', 'BookToMarket', 'InvestmentGrowth']
                       if c in df_norm.columns]
BASE_RATIO_COLS = ['ROE', 'ROA', 'Leverage', 'GrossMargin',
                   'OperatingMargin', 'CurrentRatio', 'DebtRatio', 'CFO_Margin']
RATIO_COLS = BASE_RATIO_COLS + NEW_CONFOUNDER_COLS
print(f"Ratio columns (base + new confounders): {RATIO_COLS}")

train_mask_n = df_norm['FiscalYear'].isin(TRAIN_YEARS)
winsor_bounds = {}

print('\n=== Winsorisation (1st–99th pct, TRAINING-SET bounds) ===')
for col in RATIO_COLS:
    if col not in df_norm.columns:
        continue
    lo = float(df_norm.loc[train_mask_n, col].quantile(0.01))
    hi = float(df_norm.loc[train_mask_n, col].quantile(0.99))
    n_clipped = int(((df_norm[col] < lo) | (df_norm[col] > hi)).sum())
    df_norm[col] = df_norm[col].clip(lo, hi)
    winsor_bounds[col] = {'lo': lo, 'hi': hi}
    print(f'  {col:<22}  clipped {n_clipped:4d}  |  range after: [{lo:.3f}, {hi:.3f}]')

# Persist winsorisation bounds
with open('winsor_bounds.json', 'w') as f:
    json.dump(winsor_bounds, f, indent=2)
print('\nWinsorisation bounds saved to winsor_bounds.json')

valid_ratio_cols = [c for c in RATIO_COLS if c in df_norm.columns]
print('\nRatio statistics after leakage-free winsorisation:')
df_norm[valid_ratio_cols].describe().round(3)


In [ ]:

scaler = StandardScaler()
scaler.fit(df_norm.loc[train_mask_n, valid_ratio_cols])   # FIT on train only
X_scaled = scaler.transform(df_norm[valid_ratio_cols])    # TRANSFORM all rows

df_scaled = df_norm.copy()
df_scaled[valid_ratio_cols] = X_scaled

print('=== Z-score standardisation (scaler fitted on FY2020-2022 only) ===')
print('Training-set statistics (should be ~mean=0, std=1):')
print(df_scaled.loc[train_mask_n, valid_ratio_cols].describe().round(3))

# Scaler parameters for auditability / inverse-transform
scaler_params = pd.DataFrame({
    'feature':    valid_ratio_cols,
    'mean_train': scaler.mean_.round(6),
    'std_train':  scaler.scale_.round(6)
})
scaler_params.to_csv('scaler_params.csv', index=False)
print('\nScaler parameters saved to scaler_params.csv')
print(scaler_params.to_string(index=False))


In [ ]:
import matplotlib.gridspec as gridspec

# ── Distribution plots: after leakage-free standardisation ───────────────────
fig = plt.figure(figsize=(16, max(10, len(valid_ratio_cols) * 1.3)))
n_cols_plot = 4
n_rows_plot = (len(valid_ratio_cols) + n_cols_plot - 1) // n_cols_plot
gs = gridspec.GridSpec(n_rows_plot, n_cols_plot, figure=fig)

for idx, col in enumerate(valid_ratio_cols):
    ax = fig.add_subplot(gs[idx // n_cols_plot, idx % n_cols_plot])
    ax.hist(df_scaled[col].dropna(), bins=60, color='steelblue', alpha=0.8, edgecolor='none')
    ax.set_title(col, fontsize=9)
    ax.set_ylabel('Count')
    ax.axvline(0, color='red', linewidth=0.8, linestyle='--')

fig.suptitle('Ratio distributions after leakage-free winsorisation + z-score standardisation', fontsize=11)
plt.tight_layout()
plt.show()
print('Red dashed line = zero (expected mean after standardisation)')


## 9. Save Pre-processed Dataset with Standardised Ratios

The standardised ratio columns are saved alongside the original columns.  
A `Split` column is added to tag each row as `train` / `val` / `test` for downstream modelling pipelines.

In [ ]:
# Attach standardised ratio columns to the cleaned dataset
# (raw statement columns kept as-is; only ratio columns are standardised)
df_final = df_imputed.copy()
for col in valid_ratio_cols:
    df_final[col + '_std'] = df_scaled[col]  # z-scored version
    df_final[col]          = df_norm[col]    # winsorised (not z-scored) version


df_final['Split'] = 'train'
df_final.loc[df_final['FiscalYear'].isin(VAL_YEARS),  'Split'] = 'val'
df_final.loc[df_final['FiscalYear'].isin(TEST_YEARS), 'Split'] = 'test'

df_final.to_csv('simfin_preprocessed.csv', index=False)
print(f'Saved simfin_preprocessed.csv')
print(f'Shape: {df_final.shape}')
print(f'\nSplit distribution:')
print(df_final['Split'].value_counts().to_string())
print(f'\nStandardised columns: {[c for c in df_final.columns if c.endswith("_std")]}')

# ── Leakage verification ──────────────────────────────────────────────────────
# Training-set scaler parameters should DIFFER from full-dataset parameters.
# If they are identical, the split was not applied before fitting.
scaler_full = StandardScaler().fit(df_norm[valid_ratio_cols])
check = pd.DataFrame({
    'feature':    valid_ratio_cols,
    'mean_train': scaler.mean_,
    'mean_full':  scaler_full.mean_,
    'identical':  np.isclose(scaler.mean_, scaler_full.mean_)
})
print('\nLeakage check — train vs. full-dataset scaler means (should differ):')
print(check.to_string(index=False))
if not check['identical'].all():
    print('PASS: Training-set and full-dataset means differ — no look-ahead bias.')
else:
    print('WARNING: Means are identical — check whether split was applied.')


In [ ]:
df_imputed.columns
